# Tanka 03: Helm charts and Kustomize inside Tanka

`tk tool charts` vendors charts into the project; `tanka-util` exposes `helm.template()` and
`kustomize.build()` as Jsonnet functions. The output is data, so an environment can patch it
like any other object.


In [ ]:
cd /source/work/tanka-lab
export HOME=/tmp
tk tool charts init >/dev/null 2>&1 || true
tk tool charts add-repo podinfo https://stefanprodan.github.io/podinfo && tk tool charts add podinfo/podinfo@6.15.0 && cat chartfile.yaml && ls charts


In [ ]:
cd /source/work/tanka-lab
jb install github.com/grafana/jsonnet-libs/tanka-util@master >/dev/null 2>&1 && ls vendor/github.com/grafana/jsonnet-libs/tanka-util


In [ ]:
cd /source/work/tanka-lab
cat > environments/default/main.jsonnet <<'JSONNET'
local web = import 'web.libsonnet';
local tanka = import 'github.com/grafana/jsonnet-libs/tanka-util/main.libsonnet';
local helm = tanka.helm.new(std.thisFile);

{
  _config:: { name: 'web', env: 'lab', image: 'traefik/whoami:v1.11.0', replicas: 2 },
  web: web.new($._config),

  podinfo_:: helm.template('podinfo', '../../charts/podinfo', {
    namespace: 'lab',
    values: { replicaCount: 1, ui: { message: 'hello from tanka' } },
  }),
  // helm test hooks get random names; drop them, then patch what remains
  podinfo: {
    [k]: $.podinfo_[k]
    for k in std.objectFields($.podinfo_)
    if !std.objectHas(std.get($.podinfo_[k].metadata, 'annotations', {}), 'helm.sh/hook')
  } + {
    deployment_podinfo+: { spec+: { replicas: 3 } },
  },
}
JSONNET
tk show environments/default --dangerous-allow-redirect | yq 'select(.kind == "Deployment") | .metadata.name + " replicas=" + (.spec.replicas | tostring)'


The chart came in as an object keyed by `<kind>_<name>`; one `+:` line changed the replica count without touching the chart. Kustomize works the same way through `kustomize.build`.


In [ ]:
cd /source/work/tanka-lab
mkdir -p environments/default/kustomize && cat > environments/default/kustomize/kustomization.yaml <<'YAML'
resources:
  - configmap.yaml
namePrefix: lab-
YAML
cat > environments/default/kustomize/configmap.yaml <<'YAML'
apiVersion: v1
kind: ConfigMap
metadata:
  name: motd
data:
  MOTD: rendered by kustomize inside tanka
YAML
cat >> environments/default/main.jsonnet <<'JSONNET'
+ {
  local kustomize = tanka.kustomize.new(std.thisFile),
  extras: kustomize.build('./kustomize'),
}
JSONNET
tk show environments/default --dangerous-allow-redirect | yq 'select(.kind == "ConfigMap") | .metadata.name'
